# Blackjack CV — Día 2: Dataset y Entrenamiento del Modelo

**Fecha:** 14-15 de mayo de 2026  
**Hardware:** Raspberry Pi 5 + Pi Camera Module 3 (IMX708)  
**GPU entrenamiento:** Tesla T4, 14913 MiB (Google Colab Pro)  
**Objetivo del día:** Configurar la cámara, capturar el dataset de cartas, auto-anotarlo y entrenar el modelo YOLOv8.

---

## Resumen de lo conseguido

| Paso | Estado | Resultado |
|------|--------|-----------|
| Configuración de cámara | ✅ | IMX708 funcionando con picamera2 |
| Captura de dataset | ✅ | 1450 fotos, 14 clases |
| Auto-anotación YOLO | ✅ | 1164 train / 286 val, 0 fallos |
| Entrenamiento YOLOv8n | ✅ | **mAP50 = 0.960** (mejor época: 82) |
| Velocidad de inferencia | ✅ | **3.9 ms por imagen** en T4 |

---

## 1. Configuración de la Cámara

### Hardware — Sony IMX708 (Pi Camera Module 3)

| Propiedad | Valor |
|-----------|-------|
| Resolución máxima | 4608 × 2592 px (12.3 MP) |
| Tamaño de píxel | 1.4 µm × 1.4 µm |
| Autofoco | Phase Detection AF continuo |
| Modo 0 | 1536×864 @ 120 fps |
| Modo 1 | 2304×1296 @ 56 fps |
| Modo 2 | 4608×2592 @ 14 fps |
| Modo usado | **1280×720** (escalado desde modo 0) |

### Problema 1: cv2.VideoCapture no funciona en Raspberry Pi 5

La implementación original usaba `cv2.VideoCapture(0)` de OpenCV. En Raspberry Pi 5 con Raspberry Pi OS Bookworm, la Pi Camera usa el stack **libcamera** y OpenCV no puede leer frames a través de él.

**Síntoma:** La cámara se abría sin error pero `cap.read()` devolvía siempre `ok=False`.  
**Solución:** Reemplazar por **picamera2**, la librería oficial preinstalada en Bookworm.

In [ ]:
# ANTES — no funcionaba en Pi 5
# cap = cv2.VideoCapture(0)
# ok, frame = cap.read()   # ok siempre False → RuntimeError

# DESPUÉS — solución correcta para Pi 5 + Bookworm
from picamera2 import Picamera2

cam = Picamera2()
cfg = cam.create_video_configuration(main={"format": "RGB888", "size": (1280, 720)})
cam.configure(cfg)
cam.start()
frame = cam.capture_array("main")   # array numpy (H, W, 3) en BGR
print(f"Frame OK: shape={frame.shape}  dtype={frame.dtype}")
cam.stop()
cam.close()

### Problema 2: Colores invertidos

Con el formato `BGR888`, los colores aparecían invertidos (los rojos se veían azules).

**Diagnóstico empírico:** Se capturó el mismo frame en 3 versiones y se compararon visualmente:

| Versión | Transformación | Resultado |
|---------|----------------|-----------|
| 1 — raw | Ninguna | **Colores correctos** ✅ |
| 2 — convertida | `cv2.COLOR_RGB2BGR` | Colores incorrectos ❌ |
| 3 — espejada | `cv2.flip(frame, 1)` | Orientación incorrecta ❌ |

**Conclusión:** Aunque el formato configurado es `RGB888`, picamera2 en Pi 5 devuelve datos en orden **BGR** internamente. No se necesita ninguna conversión.

In [ ]:
# Método read() final en src/perception/camera.py

# INCORRECTO — empeora los colores
# return cv2.cvtColor(cam.capture_array("main"), cv2.COLOR_RGB2BGR)

# CORRECTO — el frame raw ya es BGR en Pi 5
# return cam.capture_array("main")

### Configuración final — controles del sensor IMX708

In [ ]:
# Controles activados justo después de cam.start()
controles = {
    "AfMode":    2,     # Autofoco CONTINUO (0=manual, 1=disparo único, 2=continuo)
    "AfRange":   0,     # Rango normal      (0=normal, 1=macro, 2=full)
    "AfSpeed":   1,     # Respuesta rápida  (0=normal, 1=fast)
    "AeEnable":  True,  # Exposición automática
    "AwbEnable": True,  # Balance de blancos automático
}
# AfMode=2 (continuo) es esencial para el dataset:
# la cámara reenfoca sola cada vez que cambia la distancia o el ángulo de la carta.

---

## 2. Diseño del Dataset

### Las 14 clases

El modelo detecta el **rango** de la carta, no el palo. En blackjack el palo no influye en la estrategia.

| Clase | Tecla | Descripción |
|-------|-------|-------------|
| A | `A` | As |
| 2–9 | `2`–`9` | Números |
| 10 | `0` | Diez |
| J | `J` | Jota |
| Q | `W` | Reina (no `Q` — conflicto con salir) |
| K | `K` | Rey |
| BACK | `B` | Dorso |

**Bug corregido:** La tecla `q` estaba asignada simultáneamente a salir del script y a seleccionar la Reina. La Reina era imposible de fotografiar. Se cambió: salir → `ESC`, Reina → `W`.

### Decisión sobre el fondo de captura

La mesa real tiene un tapiz verde con patrones y texto. `auto_annotate.py` detecta cartas por análisis de contornos — ese tapiz genera bordes falsos y confunde al detector.

**Solución:** tela oscura sobre el tapiz durante la captura. El modelo YOLO aprende a reconocer cartas independientemente del fondo, por lo que funcionará sobre el tapiz real en producción.

### Técnica de captura

Para cada carta: **≥100 fotos** variando ángulo (0°–40°), posición, distancia, rotación y oclusión parcial. Se incluyeron fotos con dos cartas solapadas — reproduce exactamente cómo el crupier coloca las cartas en la mesa real.

In [ ]:
from pathlib import Path

RAW_DIR = Path("../data/raw_images")

if RAW_DIR.exists():
    total = 0
    print(f"{'Clase':>6}  {'Fotos':>6}")
    print("-" * 16)
    for clase in sorted(RAW_DIR.iterdir()):
        n = len(list(clase.glob("*.jpg")))
        total += n
        print(f"{clase.name:>6}  {n:>6}  {'█' * min(n // 5, 20)}")
    print("-" * 16)
    print(f"{'TOTAL':>6}  {total:>6}")
else:
    print("Dataset sesión 2026-05-14: 14 clases × ~103 fotos = 1450 fotos totales")

---

## 3. Auto-anotación

YOLO necesita para cada imagen un archivo `.txt` con las coordenadas del objeto en formato normalizado:

```
class_id  x_centro  y_centro  ancho  alto
```

Todos los valores excepto `class_id` son fracciones 0–1 relativas al tamaño de la imagen. `auto_annotate.py` genera estos archivos automáticamente usando detección de contornos (Canny + Otsu), válida contra la proporción estándar de una carta (63.5mm / 88.9mm ≈ 0.714) y divide el resultado en train/val.

**Resultado:** 1450 fotos anotadas, **0 fallos**, split 80/20 → 1164 train / 286 val.

---

## 4. Entrenamiento — YOLOv8 nano

### Arquitectura del modelo

**YOLOv8 nano** (`yolov8n`) es la variante más ligera de YOLOv8, diseñada para dispositivos con recursos limitados:

| Propiedad | Valor |
|-----------|-------|
| Capas totales | 130 (73 fusionadas en inferencia) |
| Parámetros | 3.013.578 |
| Operaciones | 8.2 GFLOPs |
| Pesos base | `yolov8n.pt` (preentrenado en COCO, 80 clases) |
| Clases adaptadas | 14 (nuestras cartas) |
| Pesos transferidos | 319 / 355 capas |

El modelo parte de pesos preentrenados en COCO (ImageNet) — ya sabe detectar formas y bordes genéricos. Solo hay que enseñarle las cartas específicamente. Esto se llama **transfer learning** y explica por qué aprende tan rápido con relativamente pocas fotos.

### Parámetros de entrenamiento

| Parámetro | Valor | Razón |
|-----------|-------|-------|
| `epochs` | 100 | Máximo de iteraciones |
| `patience` | 20 | Para si no mejora en 20 épocas consecutivas |
| `imgsz` | 640 | Resolución de entrada estándar de YOLO |
| `batch` | 16 | Fotos procesadas a la vez |
| `degrees` | 15 | Augmentation: rotación ±15° |
| `scale` | 0.4 | Augmentation: cambio de tamaño ±40% |
| `fliplr` | 0.5 | Augmentation: volteo horizontal 50% |
| `mosaic` | 0.5 | Augmentation: combina 4 fotos en 1 (50%) |
| `close_mosaic` | 10 | Desactiva mosaic en las últimas 10 épocas |
| GPU | Tesla T4 | Google Colab Pro |

### Curva de entrenamiento completa — 100 épocas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Datos reales extraídos del log de entrenamiento (Google Colab, 2026-05-15)
epocas = list(range(1, 101))

map50 = [
    0.167, 0.284, 0.350, 0.351, 0.467, 0.395, 0.504, 0.537, 0.524, 0.521,
    0.627, 0.587, 0.656, 0.739, 0.683, 0.740, 0.762, 0.783, 0.823, 0.819,
    0.766, 0.832, 0.868, 0.891, 0.881, 0.897, 0.895, 0.878, 0.919, 0.882,
    0.885, 0.909, 0.916, 0.932, 0.912, 0.899, 0.905, 0.903, 0.925, 0.910,
    0.930, 0.922, 0.937, 0.938, 0.930, 0.931, 0.921, 0.938, 0.948, 0.948,
    0.938, 0.959, 0.945, 0.941, 0.955, 0.951, 0.941, 0.940, 0.955, 0.958,
    0.953, 0.943, 0.953, 0.947, 0.949, 0.955, 0.955, 0.962, 0.955, 0.957,
    0.962, 0.952, 0.951, 0.959, 0.952, 0.951, 0.956, 0.952, 0.954, 0.952,
    0.957, 0.964, 0.958, 0.956, 0.957, 0.957, 0.942, 0.961, 0.953, 0.960,
    0.954, 0.951, 0.953, 0.952, 0.957, 0.955, 0.954, 0.952, 0.953, 0.955,
]

box_loss = [
    1.134, 0.942, 0.898, 0.916, 0.863, 0.863, 0.866, 0.814, 0.831, 0.796,
    0.791, 0.791, 0.788, 0.740, 0.756, 0.727, 0.773, 0.758, 0.724, 0.725,
    0.731, 0.715, 0.707, 0.676, 0.659, 0.680, 0.686, 0.673, 0.671, 0.641,
    0.635, 0.652, 0.613, 0.594, 0.577, 0.538, 0.533, 0.528, 0.522, 0.520,
    0.501, 0.478, 0.479, 0.464, 0.461, 0.452, 0.445, 0.459, 0.434, 0.441,
    0.444, 0.409, 0.430, 0.402, 0.408, 0.430, 0.393, 0.405, 0.402, 0.388,
    0.390, 0.391, 0.391, 0.381, 0.373, 0.374, 0.378, 0.377, 0.361, 0.356,
    0.358, 0.369, 0.345, 0.350, 0.349, 0.350, 0.346, 0.326, 0.314, 0.343,
    0.331, 0.325, 0.316, 0.309, 0.331, 0.315, 0.315, 0.317, 0.324, 0.300,
    0.197, 0.189, 0.187, 0.188, 0.183, 0.179, 0.174, 0.176, 0.172, 0.169,
]

cls_loss = [
    4.024, 3.278, 2.890, 2.680, 2.431, 2.248, 2.192, 2.076, 1.986, 1.879,
    1.784, 1.766, 1.672, 1.605, 1.577, 1.525, 1.486, 1.406, 1.348, 1.341,
    1.335, 1.246, 1.204, 1.183, 1.162, 1.161, 1.106, 1.101, 1.078, 1.028,
    1.042, 1.023, 0.997, 1.009, 0.947, 0.939, 0.897, 0.907, 0.859, 0.868,
    0.862, 0.871, 0.838, 0.823, 0.820, 0.827, 0.778, 0.799, 0.768, 0.765,
    0.743, 0.729, 0.712, 0.694, 0.728, 0.696, 0.675, 0.706, 0.689, 0.687,
    0.671, 0.639, 0.625, 0.659, 0.648, 0.654, 0.650, 0.621, 0.608, 0.607,
    0.596, 0.598, 0.602, 0.569, 0.593, 0.578, 0.576, 0.575, 0.520, 0.575,
    0.564, 0.556, 0.550, 0.530, 0.564, 0.535, 0.517, 0.534, 0.537, 0.503,
    0.264, 0.254, 0.250, 0.244, 0.248, 0.241, 0.236, 0.224, 0.227, 0.229,
]

mejor_epoca = epocas[map50.index(max(map50))]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Panel superior: mAP50
ax1 = axes[0]
ax1.plot(epocas, map50, color='steelblue', linewidth=1.5, label='mAP50')
ax1.axhline(y=0.90,  color='orange', linestyle='--', alpha=0.7, label='Objetivo (0.90)')
ax1.axhline(y=0.95,  color='green',  linestyle='--', alpha=0.7, label='Excelente (0.95)')
ax1.axvline(x=mejor_epoca, color='red', linestyle=':', alpha=0.7, label=f'Mejor época ({mejor_epoca})')
ax1.axvspan(91, 100, alpha=0.08, color='purple', label='close_mosaic activo')
ax1.set_ylabel('mAP50')
ax1.set_ylim(0.1, 1.02)
ax1.legend(loc='lower right', fontsize=8)
ax1.set_title('Entrenamiento YOLOv8n — Blackjack CV (100 épocas, Tesla T4)')
ax1.grid(True, alpha=0.3)
ax1.annotate(f'Mejor: {max(map50):.3f}\nÉpoca {mejor_epoca}',
             xy=(mejor_epoca, max(map50)), xytext=(mejor_epoca+3, max(map50)-0.06),
             arrowprops=dict(arrowstyle='->', color='red'), fontsize=8, color='red')

# Panel inferior: pérdidas
ax2 = axes[1]
ax2.plot(epocas, box_loss, color='tomato',     linewidth=1.5, label='box_loss')
ax2.plot(epocas, cls_loss, color='darkorange',  linewidth=1.5, label='cls_loss')
ax2.axvspan(91, 100, alpha=0.08, color='purple')
ax2.axvline(x=91, color='purple', linestyle=':', alpha=0.5)
ax2.annotate('close_mosaic\n(época 91)', xy=(91, 0.8), xytext=(80, 1.2),
             arrowprops=dict(arrowstyle='->', color='purple'), fontsize=8, color='purple')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Análisis por fases del entrenamiento

#### Fase 1 — Aprendizaje rápido (épocas 1–19)

El modelo parte de pesos preentrenados en COCO pero sin saber nada de cartas. La `cls_loss` (error de clasificación) cae dramáticamente de **4.024 a 1.348** — el modelo aprende muy deprisa a distinguir las clases. El mAP50 pasa de 0.167 a 0.823 en solo 19 épocas.

#### Fase 2 — Consolidación (épocas 20–50)

El aprendizaje se vuelve más gradual. El mAP50 supera 0.90 por primera vez en la **época 29** (0.919). Las pérdidas siguen bajando pero más despacio — el modelo ya sabe reconocer las cartas y ahora afina los detalles.

#### Fase 3 — Convergencia (épocas 51–90)

El mAP50 se estabiliza en el rango **0.94–0.96**. Las mejoras son pequeñas y no lineales — hay épocas que bajan ligeramente antes de subir. El modelo alcanza su pico en la **época 82 con mAP50=0.964**.

#### Fase 4 — close_mosaic (épocas 91–100)

A la época 91 ocurre algo llamativo: las pérdidas caen en picado de golpe (`box_loss`: 0.300 → 0.197, `cls_loss`: 0.503 → 0.264). Esto **no es un error** — es el parámetro `close_mosaic=10` funcionando. YOLOv8 desactiva la augmentación mosaic en las últimas 10 épocas para que el modelo «aterrice» sobre imágenes reales y no combinaciones artificiales. El mAP50 se mantiene estable, lo que confirma que el modelo generaliza bien.

---

## 5. Resultados de Evaluación

Evaluación del modelo `best.pt` (época 82) sobre las **286 imágenes de validación**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Resultados reales del log de evaluación (Google Colab, 2026-05-15)
clases     = ['A',  '2',   '3',   '4',   '5',   '6',  '7',   '8',   '9',
               '10', 'J',   'Q',   'K',   'BACK']
precision  = [0.954, 0.884, 1.000, 0.736, 1.000, 0.930, 0.966, 0.989, 0.997,
               0.857, 1.000, 0.917, 0.964, 0.990]
recall     = [0.935, 0.761, 0.900, 0.952, 0.745, 0.950, 0.909, 0.850, 0.850,
               0.900, 0.923, 0.950, 0.950, 1.000]
map50_cls  = [0.953, 0.925, 0.959, 0.950, 0.925, 0.940, 0.988, 0.939, 0.983,
               0.960, 0.993, 0.938, 0.993, 0.995]

x = np.arange(len(clases))
w = 0.25

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Gráfico 1: Precision y Recall por clase
ax = axes[0]
bars1 = ax.bar(x - w/2, precision, w, label='Precision', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, recall,    w, label='Recall',    color='tomato',    alpha=0.85)
ax.axhline(y=0.90, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(clases)
ax.set_ylabel('Valor')
ax.set_ylim(0.6, 1.05)
ax.set_title('Precision y Recall por clase')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Gráfico 2: mAP50 por clase
ax2 = axes[1]
colores = ['green' if v >= 0.96 else 'steelblue' if v >= 0.93 else 'orange' for v in map50_cls]
bars = ax2.bar(x, map50_cls, color=colores, alpha=0.85)
ax2.axhline(y=0.960, color='red', linestyle='--', alpha=0.6, label=f'Media global (0.960)')
ax2.set_xticks(x)
ax2.set_xticklabels(clases)
ax2.set_ylabel('mAP50')
ax2.set_ylim(0.85, 1.02)
ax2.set_title('mAP50 por clase  (verde ≥ 0.96 | azul ≥ 0.93 | naranja < 0.93)')
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, map50_cls):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.3f}',
             ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

### Métricas globales

In [ ]:
resultados = {
    "mAP50":              0.960,
    "mAP50-95":           0.949,
    "Precision (media)": 0.942,
    "Recall (media)":    0.898,
    "Mejor época":        82,
    "Épocas totales":     100,
}

velocidad = {
    "Preprocesado":   "3.1 ms",
    "Inferencia":     "3.9 ms",
    "Postprocesado":  "2.7 ms",
    "TOTAL por imagen": "9.7 ms  (~103 fps en T4)",
}

print("=== MÉTRICAS DE CALIDAD ===")
for k, v in resultados.items():
    print(f"  {k:25s}: {v}")

print()
print("=== VELOCIDAD (Tesla T4) ===")
for k, v in velocidad.items():
    print(f"  {k:25s}: {v}")

### Análisis por clase

#### Clases con mejor rendimiento

| Clase | mAP50 | Observación |
|-------|-------|-------------|
| BACK | 0.995 | El dorso es visualmente único — detección casi perfecta |
| J / K | 0.993 | Las figuras tienen símbolos muy distintivos |
| 7 | 0.988 | El 7 tiene una forma poco ambigua |
| 9 | 0.983 | Muy pocas confusiones con el 6 |

#### Clases con margen de mejora

| Clase | mAP50 | Problema específico |
|-------|-------|---------------------|
| 2 | 0.925 | Recall bajo (0.761) — el modelo no detecta algunos 2 |
| 5 | 0.925 | Recall bajo (0.745) — el modelo no detecta algunos 5 |
| Q | 0.938 | La reina se confunde puntualmente con J o K |
| 8 | 0.939 | Recall 0.850 — algunas pérdidas en ángulos extremos |

El **2** y el **5** son los más problemáticos en términos de recall. Visualmente pueden confundirse con el **7** o el **6** respectivamente según el ángulo y el estado de la carta. La solución es capturar más fotos de esas dos cartas específicas con mayor variedad de ángulos.

#### ¿Por qué BACK es casi perfecto?

El dorso de la carta es visualmente muy distinto a cualquier frente — patrón decorativo, colores diferentes, sin números ni figuras. Es la clase más fácil de detectar para el modelo.

---

## 6. Conclusiones

### Lo aprendido

**Hardware/software:**
- En Pi 5 + Bookworm, `picamera2` es obligatorio — `cv2.VideoCapture` no funciona con libcamera
- El orden de canales de picamera2 debe verificarse empíricamente, no asumirse por el nombre del formato
- El autofoco continuo del IMX708 es esencial para obtener fotos nítidas sin intervención manual

**Dataset:**
- La calidad del dato supera a la cantidad: 1450 fotos bien tomadas logran mAP50=0.960
- Incluir casos difíciles (solapamientos, oclusiones) en el dataset mejora el rendimiento real
- El fondo de captura afecta directamente a la auto-anotación — fondo contrastante es imprescindible

**Modelo:**
- YOLOv8 nano es rápido, ligero y suficientemente preciso para Raspberry Pi
- Transfer learning: partir de pesos preentrenados acelera enormemente el aprendizaje
- La caída de loss en época 91 (close_mosaic) es comportamiento normal y esperado
- Un mAP50 de 0.960 con solo una baraja es un resultado excelente para un primer entrenamiento

### Próximos pasos

1. ✅ Modelo `yolov8n_blackjack.pt` copiado a `models/` en la Raspberry Pi
2. ⏳ Calibrar colores de fichas con `scripts/calibrate_chips.py`
3. ⏳ Verificar zonas de detección con `scripts/test_camera.py`
4. ⏳ Arrancar sistema completo con `python main.py`
5. ⏳ Fine-tuning con más fotos del 2 y el 5, y fotos sobre el tapiz real